In [1]:
import requests
import pandas as pd
import time
from tqdm import tqdm


In [2]:
EMAIL = 's244771@dtu.dk'
BASE_URL_AUTHORS = 'https://api.openalex.org/authors'
BASE_URL_WORKS = 'https://api.openalex.org/works'
KEY = 'YU1NMyYLC0SJYghyehfP0i'

### Retrieve authors from website

In [3]:
df = pd.read_csv('authors.csv')  # adjust path if needed

names = set()
for entry in df['author'].dropna():
    for name in entry.split(','):
        name = name.strip()
        if name:
            names.add(name)

names = list(names)

print(f'Total unique authors: {len(names)}')

Total unique authors: 1565


### Search each name in OpenAlex and retrieve author ID

In [4]:
names = names[:10]

author_ids = []
for name in tqdm(names):
    counter = 0
    params = {
        'search': name,
        'mailto': EMAIL,
        'api_key': KEY
    }
    response = requests.get('https://api.openalex.org/authors', params=params).json()
    result = response.get('results', [])
    if result:
        author_ids.append(result[0]['id'])


100%|██████████| 10/10 [00:06<00:00,  1.58it/s]


### Retrieve papers

In [5]:
all_works = []
for index in tqdm(range(len(names)-1)):
    params = {
        'filter': f'author.id:{author_ids[index]}, author.work_count:>5, author.work_count: < 5000, work.citation_count:>10, work.authorcount:<10, ',
        'mailto': EMAIL,
        'api_key': KEY

    }
    response = requests.get(BASE_URL_WORKS, params=params).json()
    for work in response.get('results', []):
        all_works.append({
            'id': work.get('id'),
            'publication_year': work.get('publication_year'),
            'cited_by_count': work.get('cited_by_count'),
            'author_ids': [a['author']['id'] for a in work.get('authorships', [])],
            'title': work.get('title'),
            'abstract_inverted_index': work.get('abstract_inverted_index')
        })
    


100%|██████████| 9/9 [00:02<00:00,  3.79it/s]


### Data storage

In [7]:
# Convert to one DataFrame
works_df = pd.DataFrame(all_works)

# D2: IC2S2 papers
D2 = works_df[['id', 'publication_year', 'cited_by_count', 'author_ids']]
D2.to_csv('D2_papers.csv', index=False)

# D3: IC2S2 abstracts
D3 = works_df[['id', 'title', 'abstract_inverted_index']]
D3.to_csv('D3_abstracts.csv', index=False)

KeyError: "None of [Index(['id', 'publication_year', 'cited_by_count', 'author_ids'], dtype='object')] are in the [columns]"

### Filtering authors

In [ ]:
filtered_authors = authors_df[
    (authors_df['works_count'] >= 5) & 
    (authors_df['works_count'] <= 5000) &
    (authors_df['id'].notna())
]